# Pandas from First Principles
## Notebook 7: GroupBy & Aggregation

---

**Series Roadmap:**

| Notebook | Topic |
|----------|-------|
| 01 | Series Basics |
| 02 | DataFrames |
| 03 | Indexing & Selection |
| 04 | Filtering |
| 05 | Data Cleaning |
| 06 | Apply & Map |
| **07** | **GroupBy & Aggregation ← You are here** |
| 08 | Merging & Joining |

---

**What you will learn in this notebook:**
- What `groupby` is and why it's one of the most powerful tools in Pandas
- The split-apply-combine pattern
- Single and multi-column grouping
- Aggregations: `.sum()`, `.mean()`, `.count()`, `.agg()`
- `transform()` — keeping the original DataFrame shape
- Practical real-world examples with a sales dataset

---
## 1. Introduction: What is GroupBy and Why Do We Need It?

### The Excel SUMIF Analogy

If you have ever used Excel, you have probably used `SUMIF` or `SUMIFS`:  
> *"Sum all sales amounts WHERE the region = 'North'"*

Now imagine doing that for **every region at once** — North, South, East, West — and also computing the average, the count, the max... all in one line.

That is exactly what **`groupby`** does in Pandas.

---

### The Split-Apply-Combine Pattern

The whole idea behind `groupby` can be understood in three steps:

```
ORIGINAL DATA
┌──────────┬─────────┐
│ region   │ revenue │
├──────────┼─────────┤
│ North    │  85000  │
│ South    │  22000  │
│ North    │  19000  │
│ East     │  91000  │
│ South    │  78000  │
└──────────┴─────────┘
         │
    STEP 1: SPLIT  (groupby 'region')
         │
    ┌────┴────────────────┐
    │                     │
 ┌──▼───┐  ┌──────┐  ┌──▼───┐
 │North │  │South │  │ East │
 │85000 │  │22000 │  │91000 │
 │19000 │  │78000 │  │      │
 └──────┘  └──────┘  └──────┘
    │          │         │
    STEP 2: APPLY  (e.g., .sum())
    │          │         │
  104000     100000    91000
    │          │         │
    └────┬─────────────┘
         │
    STEP 3: COMBINE
         │
  ┌──────────┬─────────┐
  │ region   │ revenue │
  ├──────────┼─────────┤
  │ East     │  91000  │
  │ North    │ 104000  │
  │ South    │ 100000  │
  └──────────┴─────────┘
```

### Why Not Just Filter Manually?

You *could* do it the long way:
```python
north_total = df[df['region'] == 'North']['revenue'].sum()
south_total = df[df['region'] == 'South']['revenue'].sum()
east_total  = df[df['region'] == 'East']['revenue'].sum()
```
But this is:
- **Repetitive** — you write the same code N times
- **Not scalable** — what if there are 50 regions?
- **Hard to maintain** — adding a new region means rewriting code

With `groupby`, all of the above becomes one line:
```python
df.groupby('region')['revenue'].sum()
```

In [ ]:
# ── Setup: import pandas and create the sales dataset ──────────────────────
import pandas as pd

sales = pd.DataFrame({
    'region':      ['North','South','North','East','South','East','North','South','East','North'],
    'product':     ['Laptop','Phone','Phone','Laptop','Laptop','Phone','Tablet','Tablet','Laptop','Phone'],
    'salesperson': ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Ivy','Jack'],
    'revenue':     [85000, 22000, 19000, 91000, 78000, 18000, 32000, 29000, 88000, 21000],
    'units':       [1, 2, 2, 1, 1, 2, 3, 3, 1, 2],
    'quarter':     ['Q1','Q1','Q1','Q1','Q2','Q2','Q2','Q2','Q3','Q3'],
})

print("Sales Dataset:")
print(sales)
print(f"\nShape: {sales.shape}")

---
## 2. Basic GroupBy: Single Column

### The GroupBy Object

When you call `df.groupby('column')`, Pandas does **not** immediately compute anything.  
It returns a **GroupBy object** — think of it as a recipe, not the result.

```python
grouped = df.groupby('region')   # ← GroupBy object, nothing computed yet
result  = grouped.sum()          # ← Now the computation happens
```

You need to **apply an aggregation** to get a result.

### Common Aggregation Methods

| Method | Description |
|--------|-------------|
| `.sum()` | Total of each group |
| `.mean()` | Average of each group |
| `.count()` | Number of non-null values per group |
| `.size()` | Number of rows per group (includes NaN) |
| `.min()` | Minimum value in each group |
| `.max()` | Maximum value in each group |
| `.std()` | Standard deviation of each group |

### The Result
The result of a groupby aggregation is a **DataFrame** (or Series) where the  
group labels become the **index**.

Use `.reset_index()` to push the group labels back into regular columns.

### ⚠️ Common Mistake
```python
# WRONG — this just prints a GroupBy object description, not data
print(df.groupby('region'))

# CORRECT — always chain an aggregation
print(df.groupby('region').sum())
```

In [ ]:
# ── Inspecting the GroupBy object ──────────────────────────────────────────
grouped = sales.groupby('region')

print("Type of grouped:", type(grouped))
print("\nThis is just a GroupBy object — no data yet!")
print(grouped)

In [ ]:
# ── Applying aggregations: sum, mean, count, min, max ──────────────────────

# Total revenue and units per region
revenue_sum = sales.groupby('region')[['revenue', 'units']].sum()
print("Total revenue and units per region:")
print(revenue_sum)
print(f"\nType of result: {type(revenue_sum)}")
print(f"Note: 'region' is the INDEX, not a column")

In [ ]:
# ── reset_index(): turn the group label back into a column ─────────────────

revenue_sum_clean = sales.groupby('region')[['revenue', 'units']].sum().reset_index()

print("After reset_index() — 'region' is now a regular column:")
print(revenue_sum_clean)
print(f"\nIndex is now: {revenue_sum_clean.index.tolist()}")

In [ ]:
# ── More aggregations: mean, count, min, max ───────────────────────────────

print("Mean revenue per region:")
print(sales.groupby('region')['revenue'].mean())

print("\nCount of records per region (non-null):")
print(sales.groupby('region')['revenue'].count())

print("\nMin revenue per region:")
print(sales.groupby('region')['revenue'].min())

print("\nMax revenue per region:")
print(sales.groupby('region')['revenue'].max())

---
## 3. Selecting a Column After GroupBy

### Two Approaches — Different Results!

```python
# Approach A — select ONE column → returns a Series
df.groupby('region')['revenue'].mean()

# Approach B — no column selection → returns a DataFrame (all numeric cols)
df.groupby('region').mean()

# Approach C — select MULTIPLE columns → returns a DataFrame
df.groupby('region')[['revenue', 'units']].mean()
```

### Intuition
Think of it like this:
- **After** `groupby()` you place square brackets `[]` to choose **which columns to aggregate**
- One column in `[]` → Series result
- List of columns in `[[]]` → DataFrame result

### Why Select a Specific Column?
- Faster (less computation)
- Cleaner result — you only see what you care about
- Returns a Series, which is easier to plot directly

In [ ]:
# ── Single column selection → Series ──────────────────────────────────────
mean_revenue_series = sales.groupby('region')['revenue'].mean()

print("Single column → Series:")
print(mean_revenue_series)
print(f"\nType: {type(mean_revenue_series)}")

In [ ]:
# ── No column selection vs multiple column selection ───────────────────────

print("No column selection → DataFrame (all numeric columns):")
print(sales.groupby('region').mean(numeric_only=True))

print("\nMultiple column selection → DataFrame:")
print(sales.groupby('region')[['revenue', 'units']].mean())

---
## 4. GroupBy with Multiple Columns

### What It Does
Instead of grouping by one column, you can group by **two or more columns** simultaneously.

**Syntax:**
```python
df.groupby(['col1', 'col2']).sum()
```

### MultiIndex Result
When you group by multiple columns, the result has a **MultiIndex** — a hierarchical index with one level per grouping column.

```
                   revenue
region  quarter
East    Q1          91000   ← indexed by (East, Q1)
        Q2          18000   ← indexed by (East, Q2)
        Q3          88000
North   Q1         104000
...
```

Use `.reset_index()` to **flatten** the MultiIndex into regular columns — this is almost always what you want for further analysis or display.

### When to Group by Multiple Columns
- "Total revenue **per region** AND **per quarter**"
- "Average score **per class** AND **per subject**"
- "Headcount **per department** AND **per job level**"

In [ ]:
# ── GroupBy with two columns → MultiIndex result ───────────────────────────
multi_group = sales.groupby(['region', 'quarter'])['revenue'].sum()

print("Total revenue by region AND quarter (MultiIndex):")
print(multi_group)
print(f"\nIndex type: {type(multi_group.index)}")

In [ ]:
# ── reset_index() to flatten the MultiIndex ────────────────────────────────
region_quarter_revenue = (
    sales.groupby(['region', 'quarter'])['revenue']
         .sum()
         .reset_index()
)

print("After reset_index() — flat, clean DataFrame:")
print(region_quarter_revenue)
print(f"\nColumns: {region_quarter_revenue.columns.tolist()}")

---
## 5. `agg()` — Multiple Aggregations at Once

### The Problem With Chaining
What if you need the **mean AND max AND count** all at once?  
Calling `.mean()`, then `.max()`, then `.count()` separately is inefficient.

**Solution:** Use `.agg()` (short for *aggregate*).

### Four Ways to Use `.agg()`

**1. Single string — same as calling the method directly:**
```python
df.groupby('region')['revenue'].agg('mean')   # same as .mean()
```

**2. List of strings — multiple functions, same column:**
```python
df.groupby('region')['revenue'].agg(['mean', 'max', 'count'])
```

**3. Dictionary — different functions for different columns:**
```python
df.groupby('region').agg({'revenue': 'mean', 'units': 'sum'})
```

**4. Named aggregations — control the output column names:**
```python
df.groupby('region').agg(
    avg_revenue=('revenue', 'mean'),
    total_units=('units', 'sum')
)
```

### Why Use Named Aggregations?
- Output column names are descriptive and meaningful
- Avoids ambiguity when the same column is aggregated multiple ways
- Cleaner, more readable code

In [ ]:
# ── agg() with a single string ─────────────────────────────────────────────
print("agg('mean') — same as .mean():")
print(sales.groupby('region')['revenue'].agg('mean'))

In [ ]:
# ── agg() with a list of functions ────────────────────────────────────────
multi_agg = sales.groupby('region')['revenue'].agg(['mean', 'max', 'min', 'count'])

print("Multiple aggregations on revenue per region:")
print(multi_agg)

In [ ]:
# ── agg() with a dictionary — different functions per column ───────────────
dict_agg = sales.groupby('region').agg(
    {'revenue': 'mean', 'units': 'sum'}
)

print("Dictionary agg — mean revenue, sum of units per region:")
print(dict_agg)

In [ ]:
# ── Named aggregations — control the output column names ───────────────────
named_agg = sales.groupby('region').agg(
    avg_revenue  = ('revenue', 'mean'),
    max_revenue  = ('revenue', 'max'),
    total_units  = ('units',   'sum'),
    num_sales    = ('revenue', 'count'),
).reset_index()

print("Named aggregations — clean, descriptive column names:")
print(named_agg)

---
## 6. `transform()` — Same Shape, Group Statistics

### The Key Difference from `.agg()`

| Method | Output Shape | Use Case |
|--------|-------------|----------|
| `.agg()` | **Smaller** — one row per group | Summarizing data |
| `.transform()` | **Same** as original DataFrame | Adding group stats back to original rows |

### When to Use `transform()`
- Adding a **group-level statistic** as a new column to the original DataFrame
- Comparing each individual row to its group's aggregate
- Normalizing values within groups

### Intuition
Imagine a classroom:
- Section A: Alice (85), Bob (90), Carol (78)
- Section B: Dave (92), Eve (88)

`transform('mean')` returns:
```
Alice  → 84.33  (Section A average)
Bob    → 84.33  (Section A average)
Carol  → 84.33  (Section A average)
Dave   → 90.0   (Section B average)
Eve    → 90.0   (Section B average)
```
Each row gets the **average of its own group** — same length as the original data!

### ⚠️ Common Mistake
```python
# WRONG — agg() returns fewer rows, can't be assigned as a new column
df['group_avg'] = df.groupby('region')['revenue'].agg('mean')  # ← alignment error!

# CORRECT — transform() returns same number of rows, aligns perfectly
df['group_avg'] = df.groupby('region')['revenue'].transform('mean')
```

In [ ]:
# ── transform() — returns same shape as original DataFrame ────────────────
# We want to add each salesperson's REGION average revenue as a new column

region_avg = sales.groupby('region')['revenue'].transform('mean')

print("transform('mean') result — same length as original (10 rows):")
print(region_avg)
print(f"\nLength: {len(region_avg)} (same as sales DataFrame: {len(sales)})")

In [ ]:
# ── Use transform() to add group mean back to the original DataFrame ───────
sales_with_avg = sales.copy()
sales_with_avg['region_avg_revenue'] = sales.groupby('region')['revenue'].transform('mean')
sales_with_avg['vs_region_avg'] = sales_with_avg['revenue'] - sales_with_avg['region_avg_revenue']

print("Each salesperson's revenue vs. their region's average:")
print(sales_with_avg[['salesperson', 'region', 'revenue', 'region_avg_revenue', 'vs_region_avg']])

In [ ]:
# ── transform() for normalization within groups ────────────────────────────
# Z-score within each region: how many std-devs is each sale from its group mean?

sales_norm = sales.copy()
group_mean = sales.groupby('region')['revenue'].transform('mean')
group_std  = sales.groupby('region')['revenue'].transform('std')

# Avoid division by zero for groups with only one member
sales_norm['revenue_zscore'] = (sales_norm['revenue'] - group_mean) / group_std.replace(0, 1)

print("Revenue z-score within each region (normalization):")
print(sales_norm[['salesperson', 'region', 'revenue', 'revenue_zscore']].round(2))

---
## 7. Practical Examples

Let us now apply everything we have learned to real-world style questions using the sales dataset.

### Example A: Sales by Region
Business Question: *"Give me a complete regional performance summary"*

### Example B: Average Score by Class Section
Education Context: *"How does each section compare on average?"*

### Example C: Department Headcount and Average Salary
HR Context: *"How many people are in each department and what do they earn on average?"*

In [ ]:
# ── Example A: Full regional performance summary ──────────────────────────
regional_summary = sales.groupby('region').agg(
    total_revenue   = ('revenue', 'sum'),
    avg_revenue     = ('revenue', 'mean'),
    max_deal        = ('revenue', 'max'),
    min_deal        = ('revenue', 'min'),
    total_units     = ('units',   'sum'),
    num_salespeople = ('salesperson', 'count'),
).reset_index()

regional_summary['avg_revenue'] = regional_summary['avg_revenue'].round(0)
print("Regional Performance Summary:")
print(regional_summary.to_string(index=False))

In [ ]:
# ── Example B: Average score by class section ─────────────────────────────
# Small standalone dataset for this example
scores = pd.DataFrame({
    'student': ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank'],
    'section': ['A','A','A','B','B','B','A','B'],
    'math':    [88, 79, 91, 95, 82, 76, 85, 90],
    'english': [72, 85, 88, 78, 91, 83, 77, 86],
})

section_avg = scores.groupby('section')[['math', 'english']].mean().round(1).reset_index()

print("Average scores by class section:")
print(section_avg)

In [ ]:
# ── Example C: Department headcount and average salary ────────────────────
# Small standalone HR dataset
employees = pd.DataFrame({
    'name':       ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Ivy'],
    'department': ['Eng','Eng','Eng','HR','HR','Sales','Sales','Sales','Eng'],
    'salary':     [95000, 88000, 102000, 62000, 58000, 71000, 68000, 74000, 91000],
    'age':        [28, 32, 35, 29, 41, 27, 33, 38, 30],
})

dept_summary = employees.groupby('department').agg(
    headcount   = ('name',   'count'),
    avg_salary  = ('salary', 'mean'),
    max_salary  = ('salary', 'max'),
    avg_age     = ('age',    'mean'),
).reset_index()

dept_summary['avg_salary'] = dept_summary['avg_salary'].round(0).astype(int)
dept_summary['avg_age']    = dept_summary['avg_age'].round(1)

print("Department Headcount and Compensation Summary:")
print(dept_summary.to_string(index=False))

---
## 🧩 Practice Questions

Use the `sales` DataFrame defined at the top of this notebook to answer the following questions.

```python
# Reminder: the sales dataset columns
# region, product, salesperson, revenue, units, quarter
```

---

**Q1 [Easy]:** Find the **total revenue** for each region.  
*(Expected: East=197000, North=157000, South=129000)*

---

**Q2 [Easy]:** Find the **average revenue** per product type.  
*(Expected: Laptop, Phone, Tablet — each with their mean)*

---

**Q3 [Easy]:** Count **how many sales records** exist per region.  
*(Tip: use `.size()` or `.count()` on any column)*

---

**Q4 [Easy]:** Find the **highest single-sale revenue** in each region.  
*(Use `.max()`)*

---

**Q5 [Medium]:** Group by **both region AND quarter**, then find the **total revenue** for each combination. Flatten the result with `reset_index()`.  
*(How did the North region do in Q1 vs Q2 vs Q3?)*

---

**Q6 [Medium]:** Use **`agg()`** to get the **mean, max, and min** of `revenue` per product — all in one call.  
*(Result should have columns: mean, max, min)*

---

**Q7 [Medium]:** Add a new column `'region_avg_revenue'` to the `sales` DataFrame.  
This column should contain the **average revenue of that salesperson's region**.  
*(Use `transform()` — do NOT use `agg()`)*

---

**Q8 [Medium]:** Which **product** had the **highest total revenue**?  
*(Hint: groupby + sum + sort_values — or use idxmax())*

In [ ]:
# Your workspace — try answering the questions here!
# Q1


In [ ]:
# Q2


In [ ]:
# Q3


In [ ]:
# Q4


In [ ]:
# Q5


In [ ]:
# Q6


In [ ]:
# Q7


In [ ]:
# Q8


---
## ✅ Answer Key

Expand each answer only after attempting the question yourself!

In [ ]:
# ── Answer Q1: Total revenue by region ────────────────────────────────────
q1_answer = sales.groupby('region')['revenue'].sum().reset_index()
q1_answer.columns = ['region', 'total_revenue']
print("Q1: Total revenue by region")
print(q1_answer)

In [ ]:
# ── Answer Q2: Average revenue per product ────────────────────────────────
q2_answer = sales.groupby('product')['revenue'].mean().round(2).reset_index()
q2_answer.columns = ['product', 'avg_revenue']
print("Q2: Average revenue per product")
print(q2_answer)

In [ ]:
# ── Answer Q3: Count of sales records per region ──────────────────────────
# Method 1: .size() — counts ALL rows (including NaN)
q3_size = sales.groupby('region').size().reset_index(name='count_size')

# Method 2: .count() on a specific column — counts non-null rows
q3_count = sales.groupby('region')['revenue'].count().reset_index()
q3_count.columns = ['region', 'count_revenue']

print("Q3 — Method 1: .size()")
print(q3_size)
print("\nQ3 — Method 2: .count() on 'revenue'")
print(q3_count)
print("\nNote: Both give the same result here since there are no NaN values.")

In [ ]:
# ── Answer Q4: Highest revenue in each region ─────────────────────────────
q4_answer = sales.groupby('region')['revenue'].max().reset_index()
q4_answer.columns = ['region', 'max_revenue']
print("Q4: Highest single-sale revenue per region")
print(q4_answer)

In [ ]:
# ── Answer Q5: Group by region AND quarter, total revenue ─────────────────
q5_answer = (
    sales.groupby(['region', 'quarter'])['revenue']
         .sum()
         .reset_index()
)
q5_answer.columns = ['region', 'quarter', 'total_revenue']
print("Q5: Total revenue by region and quarter")
print(q5_answer)

# Bonus: pivot for a clearer view
print("\nBonus — Pivoted view (region × quarter):")
print(q5_answer.pivot(index='region', columns='quarter', values='total_revenue').fillna(0).astype(int))

In [ ]:
# ── Answer Q6: agg() — mean, max, min revenue per product ─────────────────
q6_answer = (
    sales.groupby('product')['revenue']
         .agg(['mean', 'max', 'min'])
         .round(0)
         .reset_index()
)
print("Q6: Mean, max, min revenue per product")
print(q6_answer)

In [ ]:
# ── Answer Q7: Add 'region_avg_revenue' column using transform() ──────────
sales_q7 = sales.copy()
sales_q7['region_avg_revenue'] = (
    sales_q7.groupby('region')['revenue'].transform('mean').round(0)
)

print("Q7: Salesperson revenue vs. region average (using transform):")
print(sales_q7[['salesperson', 'region', 'revenue', 'region_avg_revenue']])

In [ ]:
# ── Answer Q8: Which product had the highest TOTAL revenue? ───────────────

# Method 1: groupby + sum + sort_values
product_revenue = sales.groupby('product')['revenue'].sum().reset_index()
product_revenue.columns = ['product', 'total_revenue']
product_revenue_sorted = product_revenue.sort_values('total_revenue', ascending=False)

print("Q8: Total revenue per product (sorted):")
print(product_revenue_sorted)

# Method 2: idxmax() — directly find the winner
top_product = sales.groupby('product')['revenue'].sum().idxmax()
top_revenue = sales.groupby('product')['revenue'].sum().max()
print(f"\nWinner: {top_product} with total revenue of ${top_revenue:,}")

---
## 📋 Quick Revision Table

| Concept | Syntax | Returns | Notes |
|---------|--------|---------|-------|
| GroupBy object | `df.groupby('col')` | GroupBy object | Lazy — needs aggregation |
| Single agg | `df.groupby('col').sum()` | DataFrame | Group labels as index |
| Flatten index | `.reset_index()` | DataFrame | Group label → regular column |
| Select column | `df.groupby('col')['x'].mean()` | Series | Faster, focused |
| Multi-column group | `df.groupby(['c1','c2'])` | MultiIndex result | Use `.reset_index()` to flatten |
| agg() — single | `.agg('mean')` | Series/DataFrame | Same as `.mean()` |
| agg() — list | `.agg(['mean','max'])` | DataFrame | Multiple stats at once |
| agg() — dict | `.agg({'col':'mean'})` | DataFrame | Different stat per column |
| Named agg | `.agg(avg=('col','mean'))` | DataFrame | Controls output column names |
| transform() | `.transform('mean')` | Same shape as input | Adds group stat to each row |
| Count rows | `.size()` | Series | Includes NaN rows |
| Count non-null | `.count()` | Series/DataFrame | Excludes NaN |

---
## 🎤 Interview Questions

**Q1: What is the difference between `.agg()` and `.transform()` in a groupby operation?**

> **Answer:** `.agg()` reduces each group to a single value, returning a result with fewer rows (one per group). `.transform()` returns a result with the **same number of rows** as the original DataFrame, broadcasting each group's aggregate value back to all rows belonging to that group. Use `agg()` for summaries; use `transform()` when you need to add group-level statistics as a new column.

---

**Q2: What is the difference between `.size()` and `.count()` after a groupby?**

> **Answer:** `.size()` counts the **total number of rows** in each group, including rows with `NaN` values. `.count()` counts the number of **non-null values** per group per column. If there are no missing values in the data, they give the same result. However, `.count()` can return a DataFrame (one count per column), while `.size()` always returns a Series with one count per group.

---

**Q3: What does it mean when groupby returns a MultiIndex, and how do you handle it?**

> **Answer:** A MultiIndex appears when you group by **more than one column**. The result has a hierarchical index where each level corresponds to one of the grouping columns (e.g., `(region, quarter)`). It can be confusing to work with and may cause alignment issues when assigning back to a DataFrame. The standard fix is to call `.reset_index()` after the aggregation, which promotes all index levels back into regular columns, giving you a clean, flat DataFrame.

---
## 🔜 What's Next?

### Notebook 8 — Merging & Joining

You now know how to **summarize** data within a single DataFrame.  
The next step is learning how to **combine** data from **multiple DataFrames** — like SQL JOINs.

In Notebook 8, you will learn:
- `pd.merge()` — the workhorse for combining DataFrames
- Inner, Left, Right, and Outer joins — when to use which
- Merging on one key vs. multiple keys
- `pd.concat()` — stacking DataFrames vertically or horizontally
- Handling duplicate columns after a merge
- Diagnosing merge problems (many-to-many, missing keys)

---
*Pandas from First Principles | Notebook 7 of 8+*